In [2]:
import pandas as pd
import importlib
import data_audit

importlib.reload(data_audit)


df = pd.read_csv("lifestyle_in_different_towns.csv")


# city_name: Идентификатор на града.
# country: Географски регион.
# population_density: Население на квадратен километър.
# avg_income: Среден месечен доход на домакинство в долари.
# internet_penetration: Процент на домакинства, които имат достъп до Интернет.
# avg_rent: Среден месечен наем на апартамент в долари.
# air_quality_index: Индекс на качеството на въздуха, където по-ниски стойности съответстват на по-чист въздух.
# public_transport_score: Качество на обществения транспорт, където по-високи стойности съответстват на по-висока оценка.
# happiness_score: Удовлетвореност от живота, получена след допитване до случайни граждани, където по-високи стойности съответстват на по-висока оценка.
# green_space_ratio: Процент на градската площ, покрита с паркове/зелени пространства.


In [3]:
# This if is put so that i can run all 
# cells without creating data audit every time
if False:

    data_audit.create_data_audit(df, bar_chart_cols=['country'], no_chart_cols=[], hue_column=None)

In [5]:
# Preprocessing

df_dummies = pd.get_dummies(df['country'], drop_first=True, dtype=int)
df = pd.concat([df_dummies,df.drop(columns=['country'])], axis=1)

In [6]:
print(df.columns)

Index(['Asia', 'Europe', 'North America', 'Oceania', 'South America',
       'city_name', 'population_density', 'avg_income', 'internet_penetration',
       'avg_rent', 'air_quality_index', 'public_transport_score',
       'happiness_score', 'green_space_ratio'],
      dtype='object')


In [104]:
from sklearn.decomposition import PCA

pca = PCA(n_components=1)
X = df.drop(columns=['city_name'])
transformed = pca.fit_transform(X)

explained = pca.explained_variance_ratio_.sum()
print("Total variance preserved:", explained)

Total variance preserved: 0.8501787783865153


In [105]:
pca = PCA(n_components=2)
transformed = pca.fit_transform(X)

explained = pca.explained_variance_ratio_.sum()
print("Total variance preserved:", explained)

Total variance preserved: 0.9979987240724452


We can see that the most optimal is to preserve 2 dimensions with PCA

In [ ]:
import openpyxl
import clusterization
importlib.reload(clusterization)

import numpy as np

import sklearn
import hdbscan

from importlib.metadata import version

print(version("hdbscan"))
print(sklearn.__version__)

filename = './model_report.xlsx'

wb = openpyxl.Workbook()
wb.create_sheet('ModelReport')
ws = wb['ModelReport']

ws.append(['Model', 'Scaling','PCA','t-SNE','TruncatedSVD', 'Number of variables','Hyperparams', 'Inertia', 'Silhouette', 'Silhouette Increase from base model %', 'Silhouette for test data', 'Silhouette for test data increase %','Scatter plot train', 'Dendrogram', 'Scatter plot test', 'PCA variance', 'Cross-Tabulation'])

clusterization.base_model(X, ws)

model_options = [
    'agglomerative',
    'dbscan'
]

data_preprocessing = [
    {
        'scaling': False,
        'pca': False,
        'tsne': False,
        'truncatedsvd': False
    },
    {
        'scaling': True,
        'pca': True,
        'tsne': False,
        'truncatedsvd': False
    },
    {
        'scaling': True,
        'pca': False,
        'tsne': False,
        'truncatedsvd': True
    },
    {
        'scaling': True,
        'pca': False,
        'tsne': True,
        'truncatedsvd': False
    },
    {
        'scaling': True,
        'pca': True,
        'tsne': True,
        'truncatedsvd': False
    },
]

X_without_continents = X.drop(columns=['Asia', 'Europe', 'North America', 'Oceania', 'South America'])


X_options = [
    # X,
    X.drop(columns=['Asia', 'Europe', 'North America', 'Oceania', 'South America'])
]

n_features = 2

best_model = None
best_model_score = 0

models = []

for name in model_options:
    for preprocessing in data_preprocessing:
        for data in X_options:
            for n_clusters in np.arange(1, 15):

                scaling = preprocessing['scaling']
                pca = preprocessing['pca']
                tsne = preprocessing['tsne']
                truncatedsvd = preprocessing['truncatedsvd']

                unique_name = name + f'{scaling}{pca}{tsne}{truncatedsvd}{n_clusters}'

                model, score = clusterization.clusterize(
                    X=data,
                    name=unique_name,
                    n_features=n_features,
                    scaling=scaling,
                    pca=pca,
                    tsne=tsne,
                    truncatedsvd=truncatedsvd,
                    ws=ws,
                    clusterization_distance=1,
                    n_clusters=n_clusters)

                models.append((score, model, preprocessing))

                if score > best_model_score:
                    best_model_score = score
                    best_model = model

                if name != 'agglomerative':
                    break


wb.save(filename)

In [124]:
threshold = 0.35

best_models = [(score, model, preprocessing) for score, model, preprocessing  in models
    if score >= threshold
    ]

print(best_models)

[(0.607121573159855, Pipeline(steps=[('agglomerative',
                 AgglomerativeClustering(compute_full_tree=True,
                                         linkage='average'))]), {'scaling': False, 'pca': False, 'tsne': False, 'truncatedsvd': False}), (0.5141239014690167, Pipeline(steps=[('agglomerative',
                 AgglomerativeClustering(compute_full_tree=True,
                                         linkage='average', n_clusters=3))]), {'scaling': False, 'pca': False, 'tsne': False, 'truncatedsvd': False}), (0.3607298357579044, Pipeline(steps=[('agglomerative',
                 AgglomerativeClustering(compute_full_tree=True,
                                         linkage='average', n_clusters=4))]), {'scaling': False, 'pca': False, 'tsne': False, 'truncatedsvd': False}), (0.4183681660990602, Pipeline(steps=[('agglomerative',
                 AgglomerativeClustering(compute_full_tree=True,
                                         linkage='average', n_clusters=5))]), {'s

We get the best performance in agglomerative models for models that don't have any preprocessing.

In [125]:
best_models = [
    (score, model, preprocessing)
    for score, model, preprocessing in best_models
    if preprocessing['pca'] == False and preprocessing['scaling'] == False and
    preprocessing['truncatedsvd'] == False and preprocessing['tsne'] == False
]

print(best_models)

[(0.607121573159855, Pipeline(steps=[('agglomerative',
                 AgglomerativeClustering(compute_full_tree=True,
                                         linkage='average'))]), {'scaling': False, 'pca': False, 'tsne': False, 'truncatedsvd': False}), (0.5141239014690167, Pipeline(steps=[('agglomerative',
                 AgglomerativeClustering(compute_full_tree=True,
                                         linkage='average', n_clusters=3))]), {'scaling': False, 'pca': False, 'tsne': False, 'truncatedsvd': False}), (0.3607298357579044, Pipeline(steps=[('agglomerative',
                 AgglomerativeClustering(compute_full_tree=True,
                                         linkage='average', n_clusters=4))]), {'scaling': False, 'pca': False, 'tsne': False, 'truncatedsvd': False}), (0.4183681660990602, Pipeline(steps=[('agglomerative',
                 AgglomerativeClustering(compute_full_tree=True,
                                         linkage='average', n_clusters=5))]), {'s

Now these models all have a good silhouette score and the one with the highest one has only 2 clusters.

In [126]:
print(best_model_score)
print(best_model)

0.607121573159855
Pipeline(steps=[('agglomerative',
                 AgglomerativeClustering(compute_full_tree=True,
                                         linkage='average'))])


In [128]:
labels = best_model.fit_predict(X)
unique_labels = np.unique(labels)
print(unique_labels)

X_with_labels = X.copy()
X_with_labels['label'] = labels



[0 1]


In [129]:
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

display(X_with_labels.groupby('label').describe())


Asia                                              Europe            \
       count      mean       std  min  25%  50%  75%  max  count      mean   
label                                                                        
0       55.0  0.945455  0.229184  0.0  1.0  1.0  1.0  1.0   55.0  0.000000   
1      245.0  0.114286  0.318809  0.0  0.0  0.0  0.0  1.0  245.0  0.244898   

                                         North America                      \
            std  min  25%  50%  75%  max         count      mean       std   
label                                                                        
0      0.000000  0.0  0.0  0.0  0.0  0.0          55.0  0.000000  0.000000   
1      0.430907  0.0  0.0  0.0  0.0  1.0         245.0  0.204082  0.403854   

                               Oceania                                     \
       min  25%  50%  75%  max   count      mean       std  min  25%  50%   
label                                                                       
0      0.0  0.0  0.0  0.0  0.0    55.0  0.000000  0.000000  0.0  0.0  0.0   
1      0.0  0.0  0.0  0.0  1.0   245.0  0.142857  0.350643  0.0  0.0  0.0   

                South America                                               \
       75%  max         count      mean       std  min  25%  50%  75%  max   
label                                                                        
0      0.0  0.0          55.0  0.054545  0.229184  0.0  0.0  0.0  0.0  1.0   
1      0.0  1.0         245.0  0.151020  0.358802  0.0  0.0  0.0  0.0  1.0   

      population_density                                                    \
                   count         mean          std     min     25%     50%   
label                                                                        
0                   55.0  9334.163636  2001.937014  6448.0  7747.0  8916.0   
1                  245.0  2734.995918  1413.228633   100.0  1647.0  2574.0   

                        avg_income                                           \
           75%      max      count         mean          std    min     25%   
label                                                                         
0      10645.5  14427.0       55.0  2448.909091   656.533186  640.0  1930.0   
1       3724.0   6161.0      245.0  2912.122449  1277.937145  480.0  1870.0   

                              internet_penetration                        \
          50%     75%     max                count       mean        std   
label                                                                      
0      2580.0  2895.0  3760.0                 55.0  72.780000   9.730867   
1      2990.0  3970.0  5720.0                245.0  74.647755  18.252380   

                                      avg_rent                           \
        min   25%   50%    75%    max    count         mean         std   
label                                                                     
0      49.7  68.8  72.2  78.15  100.0     55.0   844.545455  261.900590   
1      34.0  62.4  76.4  88.70  100.0    245.0  1038.285714  482.734187   

                                            air_quality_index              \
         min    25%     50%     75%     max             count        mean   
label                                                                       
0      220.0  640.0   870.0   985.0  1490.0              55.0  100.200000   
1      170.0  650.0  1060.0  1380.0  2430.0             245.0   64.746939   

                                                  public_transport_score  \
             std   min   25%    50%    75%    max                  count   
label                                                                      
0      25.880351  46.0  78.5  102.0  117.0  146.0                   55.0   
1      20.187879  22.0  50.0   63.0   79.0  130.0                  245.0   

                                                           happiness_score  \
            mean        std   min   25%   50%    75%   max  

Even though this model has the best silhouette score it mostly divides the countries into Asian and Other countries because of big differences like population density, air quality index and green space ratio. This doesn't really answer our question of which cities have a higher life quality.

This is why I will explore the other models and will ignore models that only divide the cities into 2 clusters.

By looking at the model report we can see that the model with 5 clusters finds well-divided clusters and after that the models start to invent new clusters with a small amount of entries just because it's forced to. This also explains why after 5/6 clusters the silhouette score drops from .42 to around .35.

So let's look at the model that uses 5 clusters.

In [134]:
best_model_score, best_model, _ = best_models[3]
print(best_model_score)
print(best_model)


0.4183681660990602
Pipeline(steps=[('agglomerative',
                 AgglomerativeClustering(compute_full_tree=True,
                                         linkage='average', n_clusters=5))])


In [135]:

labels = best_model.fit_predict(X)
unique_labels = np.unique(labels)
print(unique_labels)

X_with_labels = X.copy()
X_with_labels['label'] = labels



[0 1 2 3 4]


In [136]:
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

display(X_with_labels.groupby('label').describe())


Asia                                              Europe            \
      count      mean       std  min  25%  50%  75%  max  count      mean   
label                                                                       
0      19.0  1.000000  0.000000  1.0  1.0  1.0  1.0  1.0   19.0  0.000000   
1      99.0  0.070707  0.257639  0.0  0.0  0.0  0.0  1.0   99.0  0.222222   
2      36.0  0.916667  0.280306  0.0  1.0  1.0  1.0  1.0   36.0  0.000000   
3      91.0  0.208791  0.408697  0.0  0.0  0.0  0.0  1.0   91.0  0.395604   
4      55.0  0.036364  0.188919  0.0  0.0  0.0  0.0  1.0   55.0  0.036364   

                                         North America                      \
            std  min  25%  50%  75%  max         count      mean       std   
label                                                                        
0      0.000000  0.0  0.0  0.0  0.0  0.0          19.0  0.000000  0.000000   
1      0.417855  0.0  0.0  0.0  0.0  1.0          99.0  0.313131  0.466127   
2      0.000000  0.0  0.0  0.0  0.0  0.0          36.0  0.000000  0.000000   
3      0.491689  0.0  0.0  0.0  1.0  1.0          91.0  0.197802  0.400549   
4      0.188919  0.0  0.0  0.0  0.0  1.0          55.0  0.018182  0.134840   

                               Oceania                                        \
       min  25%  50%  75%  max   count      mean     std  min  25%  50%  75%   
label                                                                          
0      0.0  0.0  0.0  0.0  0.0    19.0  0.000000  0.0000  0.0  0.0  0.0  0.0   
1      0.0  0.0  0.0  1.0  1.0    99.0  0.353535  0.4805  0.0  0.0  0.0  1.0   
2      0.0  0.0  0.0  0.0  0.0    36.0  0.000000  0.0000  0.0  0.0  0.0  0.0   
3      0.0  0.0  0.0  0.0  1.0    91.0  0.000000  0.0000  0.0  0.0  0.0  0.0   
4      0.0  0.0  0.0  0.0  1.0    55.0  0.000000  0.0000  0.0  0.0  0.0  0.0   

           South America                                               \
       max         count      mean       std  min  25%  50%  75%  max   
label                                                                   
0      0.0          19.0  0.000000  0.000000  0.0  0.0  0.0  0.0  0.0   
1      1.0          99.0  0.040404  0.197907  0.0  0.0  0.0  0.0  1.0   
2      0.0          36.0  0.083333  0.280306  0.0  0.0  0.0  0.0  1.0   
3      0.0          91.0  0.197802  0.400549  0.0  0.0  0.0  0.0  1.0   
4      0.0          55.0  0.272727  0.449467  0.0  0.0  0.0  1.0  1.0   

      population_density                                               \
                   count          mean          std      min      25%   
label                                                                   
0                   19.0  11644.210526  1281.465852  10125.0  10645.5   
1                   99.0   1559.272727   680.014073    100.0   1011.0   
2                   36.0   8114.972222   955.561151   6448.0   7329.0   
3                   91.0   4179.505495   919.590892   2562.0   3453.5   
4                   55.0   2461.290909   805.575103    421.0   1979.5   

                                 avg_income                                   \
           50%      75%      max      count         mean         std     min   
label                                                                          
0      11310.0  12362.5  14427.0       19.0  2573.157895  620.260274  1690.0   
1       1567.0   2038.0   3461.0       99.0  3768.282828  894.651957  1760.0   
2       8149.5   8824.5   9819.0       36.0  2383.333333  674.066550   640.0   
3       3984.0   4747.0   6161.0       91.0  3031.538462  874.317414  1350.0   
4       2460.0   3030.5   3921.0       55.0  1173.454545  453.689477   480.0   

                                      internet_penetration             \
          25%     50%     75%     max                count       mean   
label                                                                   
0      2040.0  2650.0  2940.0  3760.0                 19.0  75.710526   
1      3185.0  3910.0

Now we have to find meaning in all of the clusters and why they end up the way they are.

Cluster 0:
All countries are asian.
Highest population density of ~12000
Average income ~2600
Internet penetration ~75
Average rent ~850
Air quality index ~110
Public transport score ~67
Happiness score ~5
Green space ratio ~20

Cluster 1:
Mostly Europe, North America and Oceania
Lowest pop. density of ~1600
Highest avg income ~3900
Highest internet pen ~86
Highest avg rent ~1350
Lowest air quality index ~50
Public transport score ~60
Highest happiness score ~8
High green space ratio ~37

Cluster 2:
Mostly Asian with some South American
Second highest pop. density of ~8000
Avg income ~2500
Internet pen. ~72
Avg. rent ~850
Air quality index ~100
Public transport score ~60
Happiness score ~5.6
Green space ratio ~25

Cluster 3:
Europe 40% with Asia, North America and South America 20% each
Pop. density ~4000
Avg. income ~3000
Internet pen. ~76
Avg. rent ~1050
Air quality index ~71
Public transport ~60
Happiness ~7.1
Green space ratio ~35

Cluster 4:
Africa 60%, South America 30%
Pop. density ~2500
Lowest avg. income ~1000
Lowest internet pen. ~50
Lowest rent ~350
Air quality index ~80
Lowest pub. transport ~38
Lowest happiness score ~4.6
Highest green space ration ~37


From these statistics we can deduce that:

Cluster 0: Most densely populated Asian cities, with high urbanisation and low green space ratio. Worst air quality. Overall pretty bad happiness score.

Cluster 1: Most developed European, North American and Oceanian countries. With highest quality of life and highest happiness score.

Cluster 2: Very densely populated cities mostly in Asia and some in South America. Overall very similar to Cluster 0.


Cluster 3: These are developed countries with pretty high income, high air quality and overall high happiness score, but not on the level of Cluster 1.


Cluster 4: Least developed countries, mostly poor African and South American, with lowest happiness score

Proposed index for quality of life based on the cluster labels is:

0 - Cluster 1;
0.2 - Cluster 3;
0.6 - Cluster 2;
0.7 - Cluster 0;
1 - Cluster 4;

Where lower index score means higher quality of life.